### Uvoz biblioteka

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer

# Podesavanje prikaza
print("Biblioteke su uspjesno ucitane!")

Biblioteke su uspjesno ucitane!


### Definisanje klasa 

In [2]:
CLASSES = {
    "Nucleus": "nucleus.tsv",
    "Cytoplasm": "cytoplasm.tsv",
    "Cell membrane": "cell_membrane.tsv",
    "Mitochondrion": "mitochondrion.tsv",
    "Secreted": "secreted.tsv"
}

### Učitavanje i spajanje TSV fajlova

In [6]:
all_dfs = []
for label, filename in CLASSES.items():
    df = pd.read_csv(f"../../data/raw/localization/{filename}", sep="\t")
    df["label"] = label
    print(f"{label}: {len(df)} proteina")
    all_dfs.append(df)

dataset = pd.concat(all_dfs, ignore_index=True)
print(f"\nUkupno učitano proteina: {len(dataset)}")

Nucleus: 5749 proteina
Cytoplasm: 5842 proteina
Cell membrane: 4119 proteina
Mitochondrion: 1368 proteina
Secreted: 2143 proteina

Ukupno učitano proteina: 19221


### Upoznavanje sa skupom podataka

In [7]:
# Dimenzije i kolone
print(f"Dimenzije dataseta: {dataset.shape}")
print(f"Broj klasa: {dataset['label'].nunique()}")
print(f"Kolone: {list(dataset.columns)}\n")

Dimenzije dataseta: (19221, 6)
Broj klasa: 5
Kolone: ['Entry', 'Entry Name', 'Sequence', 'Protein names', 'Keywords', 'label']



In [8]:
# Prikaz prvih redova
print("Prvih 5 redova:")
dataset.head()

Prvih 5 redova:


,Entry,Entry Name,Sequence,Protein names,Keywords,label
0,A0A0C5B5G6,MOTSC_HUMAN,MRWQEMGYIFYPRKLR,Mitochondrial-derived peptide MOTS-c (Mitochon...,DNA-binding;Mitochondrion;Nucleus;Osteogenesis...,Nucleus
1,A0A2R8Y7D0,TINCR_HUMAN,MEGLRRGLSRWKRYHIKVHLADEALLLPLTVRPRDTLSDLRAQLVG...,Ubiquitin domain-containing protein TINCR (Pla...,3D-structure;Cell junction;Cytoplasm;Nucleus;P...,Nucleus
2,A0A8I5KQE6,RPSA2_HUMAN,MSGALDVLQMKEEDVLKFLAAGTHLGGTNLDFQMEHYIYKRKSDGI...,Small ribosomal subunit protein uS2B (37 kDa l...,Acetylation;Cell membrane;Cytoplasm;Membrane;N...,Nucleus
3,A0AV96,RBM47_HUMAN,MTAEDSTAAMSSDSAAGSSAKVPEGVAGAPNEAALLALMERTGYSM...,RNA-binding protein 47 (RNA-binding motif prot...,3D-structure;Alternative splicing;Cytoplasm;Me...,Nucleus
4,A0AVK6,E2F8_HUMAN,MENEKENLFCEPHKRGLMKTPLKESTTANIVLAEIQPDFGPLTTPT...,Transcription factor E2F8 (E2F-8),3D-structure;Activator;Cell cycle;DNA-binding;...,Nucleus


In [9]:
# Provjera nedostajucih vrijednosti
print("Broj nedostajucih vrijednosti po kolonama:")
print(dataset.isnull().sum())

Broj nedostajucih vrijednosti po kolonama:
Entry            0
Entry Name       0
Sequence         0
Protein names    0
Keywords         0
label            0
dtype: int64


In [10]:
# Identifikovanje dupliranih sekvenci i multifunkcionalnih proteina
dup_count = dataset['Sequence'].duplicated(keep=False).sum()
unique_seq_count = dataset['Sequence'].nunique()

print(f"Ukupno unikatnih sekvenci: {unique_seq_count}")
print(f"Broj redova sa dupliranim sekvencama: {dup_count}")

Ukupno unikatnih sekvenci: 14587
Broj redova sa dupliranim sekvencama: 8611


### Pretprocesiranje i čišćenje sekvenici

Uklanjamo prazne redove, sekvence kraće od 10 aminokiselina i sekvence koje sadrže nestandardne aminokiseline (B, J, O, U, X, Z).

In [11]:
# Redovi sa praznom sekvencom
print(f"Broj redova sa praznom sekvencom: {dataset['Sequence'].isna().sum()}")

Broj redova sa praznom sekvencom: 0


In [12]:
def valid_sequence(seq):
    valid_aa = set("ACDEFGHIKLMNPQRSTVWY")
    seq = str(seq).upper().strip()

    unvalid = [aa for aa in seq if aa not in valid_aa]
    if unvalid:
        print(f"Nevalidni karakteri: {set(unvalid)}")
        return None
    
    if len(seq) < 10:
        print(f"Prekratka sekvenca: {len(seq)}")
        return None
    
    return seq

# Validacija
dataset['Sequence'] = dataset['Sequence'].apply(valid_sequence)

total_before = len(dataset)
dataset = dataset.dropna(subset=['Sequence'])
total_after = len(dataset)

print(f"Prije odbacivanja: {total_before} proteina")
print(f"Nakon odbacivanja: {total_after} proteina")
print(f"Uklonjeno nevalidnih sekvenci: {total_before - total_after}")

Nevalidni karakteri: {'U'}
Nevalidni karakteri: {'U'}
Nevalidni karakteri: {'U'}
Nevalidni karakteri: {'U'}
Nevalidni karakteri: {'U'}
Nevalidni karakteri: {'U'}
Nevalidni karakteri: {'U'}
Nevalidni karakteri: {'U'}
Nevalidni karakteri: {'U'}
Nevalidni karakteri: {'U'}
Nevalidni karakteri: {'U'}
Nevalidni karakteri: {'U'}
Nevalidni karakteri: {'U'}
Nevalidni karakteri: {'U'}
Nevalidni karakteri: {'U'}
Nevalidni karakteri: {'U'}
Prekratka sekvenca: 5
Prekratka sekvenca: 4
Prekratka sekvenca: 2
Nevalidni karakteri: {'U'}
Nevalidni karakteri: {'U'}
Nevalidni karakteri: {'U'}
Nevalidni karakteri: {'U'}
Nevalidni karakteri: {'U'}
Nevalidni karakteri: {'U'}
Nevalidni karakteri: {'U'}
Prekratka sekvenca: 5
Prije odbacivanja: 19221 proteina
Nakon odbacivanja: 19194 proteina
Uklonjeno nevalidnih sekvenci: 27


### Grupisanje dataseta po sekvenci

In [13]:
grouped = dataset.groupby('Sequence').agg({
    'Entry': 'first',
    'label': lambda x: sorted(set(x))
}).reset_index()

grouped['num_classes'] = grouped['label'].apply(len)
print(f"Ukupno unikatnih proteinskih sekvenci: {len(grouped)}")
print(f"\nDistribucija broja klasa po sekvenci:")
print(grouped['num_classes'].value_counts().sort_index())

Ukupno unikatnih proteinskih sekvenci: 14566

Distribucija broja klasa po sekvenci:
num_classes
1    10609
2     3402
3      494
4       56
5        5
Name: count, dtype: int64


### Formiranje Single-Class (SC) i Multi-Class (MC) skupova

Grupišemo klase po unikatnim sekvencama i dijelimo podatke na dva skupa:
- **SC (Single-Class):** Sekvence sa tačno 1 funkcionalnom klasom
- **MC (Multi-Class):** Sekvence sa 2 ili više funkcionalnih klasa

In [14]:
grouped['is_multilabel'] = grouped['num_classes'] > 1

sc_count = (~grouped['is_multilabel']).sum()
mc_count = grouped['is_multilabel'].sum()

print(f"Single-Class (SC) proteina: {sc_count}")
print(f"Multi-Class (MC) proteina:  {mc_count}")

Single-Class (SC) proteina: 10609
Multi-Class (MC) proteina:  3957


### Multi-hot enkodiranje klasa

In [15]:
mlb = MultiLabelBinarizer()
label_matrix = mlb.fit_transform(grouped['label'])
label_df = pd.DataFrame(label_matrix, columns=mlb.classes_)

print(f"Klase (redoslijed kolona u multi-hot matrici): {list(mlb.classes_)}")

dataset_final = pd.concat([grouped.reset_index(drop=True), label_df], axis=1)
dataset_final.head()

Klase (redoslijed kolona u multi-hot matrici): ['Cell membrane', 'Cytoplasm', 'Mitochondrion', 'Nucleus', 'Secreted']


,Sequence,Entry,label,num_classes,is_multilabel,Cell membrane,Cytoplasm,Mitochondrion,Nucleus,Secreted
0,AEYFQHWGQGTLVTVSS,A0A0C4DH62,"[Cell membrane, Secreted]",2,True,1,0,0,0,1
1,AKNIQYFGAGTRLSVL,A0A0A0MT87,[Cell membrane],1,False,1,0,0,0,0
2,APTKAPDVFPIISGCRHPKDNSPVVLACLITGYHPTSVTVTWYMGT...,P01880,"[Cell membrane, Secreted]",2,True,1,0,0,0,1
3,ASPTSPKVFPLSLCSTQPDGNVVIACLVQGFFPQEPLSVTWSESGQ...,P01876,"[Cell membrane, Secreted]",2,True,1,0,0,0,1
4,ASPTSPKVFPLSLDSTPQDGNVVVACLVQGFFPQEPLSVTWSESGQ...,P01877,"[Cell membrane, Secreted]",2,True,1,0,0,0,1


In [16]:
# Svaki protein sa >1 labelom doprinosi u više kolona istovremeno
print("Broj pojavljivanja po klasi (uzimajući u obzir multifunkcionalne proteine):")
print(label_df.sum().sort_values(ascending=False))

Broj pojavljivanja po klasi (uzimajući u obzir multifunkcionalne proteine):
Cytoplasm        5817
Nucleus          5735
Cell membrane    4101
Secreted         2127
Mitochondrion    1364
dtype: int64


### Čuvanje obrađenih podataka

In [18]:
processed_dir = os.path.join("..", "..", "data", "processed", "localization")
if not os.path.exists(processed_dir):
    os.makedirs(processed_dir, exist_ok=True)

# Glavni dataset za dalji rad
final_path = os.path.join(processed_dir, "dataset_final.csv")
dataset_final.to_csv(final_path, index=False)

# Sporedni fajlovi - SC vs. MC
sc_path = os.path.join(processed_dir, "df_sc_reference.csv")
mc_path = os.path.join(processed_dir, "df_mc_reference.csv")

df_sc_ref = dataset_final[~dataset_final['is_multilabel']].copy()
df_mc_ref = dataset_final[dataset_final['is_multilabel']].copy()

df_sc_ref.to_csv(sc_path, index=False)
df_mc_ref.to_csv(mc_path, index=False)

print("Podaci su uspješno sačuvani u 'data/processed/localization':")
print(f" - {final_path}  (glavni dataset, {len(dataset_final)} proteina)")
print(f" - {sc_path}  (referentni SC skup, {len(df_sc_ref)} proteina)")
print(f" - {mc_path}  (referentni MC skup, {len(df_mc_ref)} proteina)")

Podaci su uspješno sačuvani u 'data/processed/localization':
 - ..\..\data\processed\localization\dataset_final.csv  (glavni dataset, 14566 proteina)
 - ..\..\data\processed\localization\df_sc_reference.csv  (referentni SC skup, 10609 proteina)
 - ..\..\data\processed\localization\df_mc_reference.csv  (referentni MC skup, 3957 proteina)
